# Landuse Polygons

<br><br>

## 0. Setup

### 0.1. Imports and Config

In [ ]:
import shapely
import numpy as np
import pandas as pd
import geopandas as gpd
from rasterio.features import geometry_mask, rasterize

from splendor.config import CORRIDOR_RADIUS, EPSG, STUDY_AREA
from splendor.io import load_network, load_raster
from splendor.landuse.types import N_TYPES, SHORT_NAMES
from splendor.landuse.corridors import get_link_corridors
from splendor.landuse.shares import get_pixel_values
from splendor.paths import (
    BIKENODEPLANNER_DIR,
    bikenodeplanner_landuse_path,
    basemap05_main_types_path,
    network_path,
    landuse_polygon_gaps_path,
    ensure_parent
)

BOUNDARIES_PATH = BIKENODEPLANNER_DIR / "data/municipality-boundaries/municipality-boundaries.gpkg"
MUNICIPALITIES = ["0360", "0376"] # Pilot municipalities (Lolland, Guldborgsund), change if STUDY_AREA != "pilot"

### 0.2. Data

In [2]:
boundaries = gpd.read_file(BOUNDARIES_PATH).to_crs(EPSG)
boundaries = boundaries[boundaries["kommunekode"].isin(MUNICIPALITIES)]
assert len(boundaries) == len(MUNICIPALITIES), "no match: check kommunekode's dtype"

landuse = pd.concat([
    gpd.read_file(bikenodeplanner_landuse_path(code)).assign(kommunekode=code)
    for code in MUNICIPALITIES
], ignore_index=True)

_, edges = load_network(network_path(STUDY_AREA))
corridors = get_link_corridors(edges, CORRIDOR_RADIUS)
raster, transform, _ = load_raster(basemap05_main_types_path(STUDY_AREA))

<br><br>

## 1. Coverage

In [3]:
covered = landuse.union_all()

# Within the municipalities
municipality_area = boundaries.union_all()
municipality_gaps = municipality_area.difference(covered)
print(f"Municipalities: {municipality_gaps.area / municipality_area.area:.1%} not covered")

# Within the link corridors
corridor_area = corridors.union_all()
corridor_gaps = corridor_area.difference(covered)
print(f"Corridors: {corridor_gaps.area / corridor_area.area:.1%} not covered")

Municipalities: 6.8% not covered
Corridors: 12.2% not covered


In [4]:
# Compare to what Basemap05 shows in the corridor gaps
gap_values = get_pixel_values(corridor_gaps, raster, transform)
gap_counts = np.bincount(gap_values, minlength=N_TYPES + 1)[1:] # Drop no data
gap_shares = pd.Series(gap_counts / gap_counts.sum(), index=[SHORT_NAMES[k] for k in range(1, N_TYPES + 1)])
gap_shares.sort_values(ascending=False).round(3)

Road/rail       0.433
Sea             0.143
Unclassified    0.141
Cropland        0.138
Settlement      0.041
Coast           0.034
Grassland       0.023
Rivers          0.020
Lagoons         0.010
Forest          0.009
Mires           0.004
Lakes           0.004
Heathland       0.000
Sparse veg.     0.000
dtype: float64

In [5]:
# Save the gaps so we can look at them in QGIS
gaps = gpd.GeoDataFrame(
    {"zone": ["municipalities", "corridors"]},
    geometry=[municipality_gaps, corridor_gaps],
    crs=EPSG
)
gaps.to_file(ensure_parent(landuse_polygon_gaps_path(STUDY_AREA)))

<br><br>

## 2. Overlaps

In [10]:
# Total overlap, the summed polygon area minus the area of their union
overlap_area = landuse.area.sum() - covered.area
print(f"Overlap: {overlap_area / 1e6:.2f} km^2 ({overlap_area / covered.area:.2%} of covered area)")

Overlap: 60.42 km^2 (3.61% of covered area)


In [11]:
# Chech which classes overlap
pairs = gpd.sjoin(landuse, landuse, predicate="intersects")
pairs = pairs[pairs.index < pairs["index_right"]]

left = landuse.geometry.loc[pairs.index].to_numpy()
right = landuse.geometry.loc[pairs["index_right"]].to_numpy()
pairs["overlap_m2"] = shapely.area(shapely.intersection(left, right))

# Pairs that only touch have approx. 0 overlap, we keep only real overlaps
overlaps = pairs[pairs["overlap_m2"] > 1].copy()
overlaps["pair"] = [" / ".join(sorted(p)) for p in zip(overlaps["type_left"], overlaps["type_right"])]
overlaps["same_municipality"] = overlaps["kommunekode_left"] == overlaps["kommunekode_right"]

In [12]:
# Overlap per class pair in km^2, split by within vs across municipalities
(
    overlaps.groupby(["pair", "same_municipality"])["overlap_m2"].sum()
    .div(1e6).unstack(fill_value=0).round(3)
    .sort_values(True, ascending=False)
    .head(20)
)

same_municipality,True
pair,
Dyrkede afgrøder på marker / Skov,19.975
Afgræssede marker / Vådområde,8.394
Blandet / Skov,6.090
Dyrkede afgrøder på marker / Vådområde,3.369
Dyrkede afgrøder på marker / Gartneri,3.229
Sø / Vådområde,2.518
Dyrkede afgrøder på marker / Sommerhusområde,1.930
Afgræssede marker / Energiforsyningsanlæg,1.786
Skov / Sø,1.651


<br><br>

## 3. Spatial Detail

In [13]:
landuse["area_m2"] = landuse.area
landuse["n_vertices"] = shapely.get_num_coordinates(landuse.geometry.to_numpy())

# Size and complexity of polygons overall
landuse[["area_m2", "n_vertices"]].describe(percentiles=[.01, .05, .25, .5, .75, .95]).round(1)

,area_m2,n_vertices
count,40434.0,40434.0
mean,42889.9,46.1
std,175774.8,77.5
min,0.0,4.0
1%,62.3,5.0
5%,154.0,8.0
25%,1180.4,16.0
50%,5684.4,27.0
75%,31929.3,54.0
95%,205231.8,133.0


In [14]:
# Smallest mapped features, compare to the 10 m raster (one pixel = 100 m2)
for threshold_m2 in [100, 1000, 10000]:
    share = (landuse["area_m2"] < threshold_m2).mean()
    print(f"Polygons < {threshold_m2:>6,} m^2: {share:.1%}")

Polygons <    100 m^2: 2.0%
Polygons <  1,000 m^2: 23.1%
Polygons < 10,000 m^2: 60.0%


In [15]:
# Per classes, how large and how detailed
(
    landuse.groupby("type")
    .agg(n=("area_m2", "size"), median_m2=("area_m2", "median"),
         min_m2=("area_m2", "min"), median_vertices=("n_vertices", "median"))
    .sort_values("n", ascending=False)
    .round(0)
)

,n,median_m2,min_m2,median_vertices
type,,,,
Dyrkede afgrøder på marker,11303,68847.0,0.0,53.0
Sø,8301,336.0,1.0,23.0
Lav bebyggelse,7654,4240.0,0.0,17.0
Braklagte marker,5322,3604.0,1.0,26.0
Afgræssede marker,2788,10242.0,2.0,38.0
Skov,2099,8219.0,0.0,24.0
Kratbevoksning,682,191.0,8.0,16.0
Blandet,652,3075.0,0.0,14.0
Vådområde,609,17183.0,324.0,49.0


<br><br>

## 4. Classes

In [16]:
# Area per group and class
landuse.groupby(["gruppe", "type"])["area_m2"].sum().div(1e6).round(2).sort_values(ascending=False)

gruppe               type                      
Dyrkede arealer      Dyrkede afgrøder på marker    1186.78
Naturtyper           Skov                           215.06
Bymæssig anvendelse  Lav bebyggelse                  98.64
Dyrkede arealer      Afgræssede marker               73.42
                     Braklagte marker                45.50
Naturtyper           Vådområde                       30.53
                     Sø                              22.55
Bymæssig anvendelse  Sommerhusområde                 19.99
                     Erhverv                          9.33
Teknisk areal        Sportsanlæg                      6.48
Skovinddeling        Blandet                          6.09
Naturtyper           Gartneri                         5.13
Teknisk areal        Andre                            2.88
                     Energiforsyningsanlæg            2.21
Naturtyper           Sandklit                         1.92
                     Råstofområde                     1.80
Bymæssig

In [18]:
# Rasterize the polygon classes onto the Basemap05 grid (0 = no polygon)
type_names = sorted(landuse["type"].unique())
type_ids = {name: i + 1 for i, name in enumerate(type_names)}
polygon_raster = rasterize(
    zip(landuse.geometry, landuse["type"].map(type_ids)), # Where polygons overlap, the last one wins
    out_shape=raster.shape, transform=transform, fill=0, dtype="uint8"
)

# Only take pixels inside the link corridors, and with Basemap05 data
in_corridors = geometry_mask([corridor_area], out_shape=raster.shape, transform=transform, invert=True)
valid = in_corridors & (raster > 0)

In [19]:
# For each polygon class, which Basemap05 types its pixels have (row shares)
polygon_names = {0: "(no polygon)", **{i: name for name, i in type_ids.items()}}
class_table = pd.crosstab(
    pd.Series(polygon_raster[valid]).map(polygon_names).rename("polygon class"),
    pd.Series(raster[valid]).map(SHORT_NAMES).rename("Basemap05"),
    normalize="index"
)
class_table.round(2)

Basemap05,Coast,Cropland,Forest,Grassland,Heathland,Lagoons,Lakes,Mires,Rivers,Road/rail,Sea,Settlement,Unclassified
polygon class,,,,,,,,,,,,,
(no polygon),0.03,0.14,0.01,0.02,0.00,0.01,0.00,0.00,0.02,0.43,0.14,0.04,0.14
Afgræssede marker,0.03,0.51,0.00,0.41,0.00,0.00,0.00,0.01,0.01,0.02,0.00,0.00,0.00
Andre,0.00,0.00,0.00,0.04,0.00,0.00,0.03,0.00,0.00,0.47,0.00,0.45,0.00
Blandet,0.00,0.00,0.85,0.00,0.00,0.00,0.00,0.02,0.02,0.10,0.00,0.01,0.00
Braklagte marker,0.00,0.90,0.00,0.04,0.00,0.00,0.00,0.00,0.03,0.03,0.00,0.00,0.00
Bykerne,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.31,0.00,0.69,0.00
Dyrkede afgrøder på marker,0.00,0.97,0.01,0.00,0.00,0.00,0.00,0.00,0.00,0.02,0.00,0.00,0.00
Energiforsyningsanlæg,0.00,0.43,0.00,0.32,0.00,0.00,0.00,0.00,0.00,0.02,0.00,0.22,0.00
Erhverv,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.28,0.00,0.71,0.00


<br><br>

## 5. Municipality Borders

In [20]:
# For each municipality, how much of its polygon area lies outside the municipality
own_boundary = boundaries.set_index("kommunekode").geometry
own_boundaries = gpd.GeoSeries(landuse["kommunekode"].map(own_boundary), index=landuse.index, crs=EPSG)

landuse["outside_m2"] = landuse.geometry.difference(own_boundaries).area
outside_share = landuse.groupby("kommunekode")[["outside_m2", "area_m2"]].sum()
outside_share["share_outside"] = outside_share["outside_m2"] / outside_share["area_m2"]
outside_share.round(4)

,outside_m2,area_m2,share_outside
kommunekode,,,
0360,0.0,8.622600e+08,0.0
0376,0.0,8.719504e+08,0.0


In [21]:
# Exact duplicates (same geometry, possibly in both municipalities)
landuse["geom_key"] = shapely.to_wkb(shapely.normalize(landuse.geometry.to_numpy()), hex=True)
duplicates = landuse[landuse.duplicated("geom_key", keep=False)]
print(f"{len(duplicates)} polygons have an exact duplicate")
duplicates.groupby(["kommunekode", "type"]).size()

20 polygons have an exact duplicate


kommunekode  type                
0360         Skov                    3
             Sommerhusområde skov    3
0376         Skov                    7
             Sommerhusområde skov    7
dtype: int64